# <center> Проект: Анализ вакансий из HeadHunter

In [2]:
import psycopg2
import pandas as pd
import requests

print(psycopg2.__version__)
print(pd.__version__)
print(requests.__version__)

2.9.12 (dt dec pq3 ext lo64)
2.3.3
2.32.5


In [3]:
import warnings

# убрал предупреждение о рекомендуемом использовании SQLAlchemy
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy')    


In [4]:
# Данные на вход: для продолжения работы введине необходимые входные данные в соответствующие поля
connection = psycopg2.connect(
database = 'database',
user = 'user',
password = 'password',
host = '127.0.0.1',
port = 80
)

# Юнит 3. Предварительный анализ данных

### 3.1. Напишите запрос, который посчитает количество вакансий в нашей базе (вакансии находятся в таблице vacancies). 

In [5]:
query_3_1 = "SELECT COUNT(*) AS total_vacancies FROM vacancies;"

In [6]:
df = pd.read_sql_query(query_3_1, connection)
print(f"Количество вакансий: {df.iloc[0, 0]}")

Количество вакансий: 49197


### 3.2. Напишите запрос, который посчитает количество работодателей (таблица employers). 

In [7]:
query_3_2 = "SELECT COUNT(*) AS total_employers FROM employers;"

In [8]:
df = pd.read_sql_query(query_3_2, connection)
print(f"Количество работодателей: {df.iloc[0, 0]}")

Количество работодателей: 23501


### 3.3. Подсчитайте с помощью запроса количество регионов (таблица areas).

In [9]:
query_3_3 = "SELECT COUNT(*) AS total_areas FROM areas;"

In [10]:
df = pd.read_sql_query(query_3_3, connection)
print(f"Количество регионов: {df.iloc[0, 0]}")

Количество регионов: 1362


### 3.4. Подсчитайте с помощью запроса количество сфер деятельности в базе (таблица industries).

In [11]:
query_3_4 = "SELECT COUNT(*) AS total_industries FROM industries;"

In [12]:
df = pd.read_sql_query(query_3_4, connection)
print(f"Количество сфер деятельности: {df.iloc[0, 0]}")

Количество сфер деятельности: 294


### Вывод по Юниту 3:

В ходе выполнения предварительного анализа, были установлены основные характеристики базы данных. 
1. Общее количество вакансий составляет 49 197, что значительно превышает число работодателей (23 501), т.е. в среднем на одного работодателя приходится около 2,1 вакансии. 
2. Количество регионов (1 362) показывает, что база покрывает широкую географию, при этом среднее число вакансий на один регион — примерно 36, а работодателей на регион — около 17. 
3. Количество сфер деятельности (294) свидетельствует о достаточно высокой степени детализации отраслевой структуры, что позволит в дальнейшем провести анализ по отраслевому и территориальному признакам.

Полученные данные создают основу для дальнейшей аналитической работы и позволяют перейти к более глубокому исследованию структуры рынка труда на основе имеющейся информации.

*** 

# Юнит 4. Детальный анализ вакансий

### 4.1. Напишите запрос, который позволит узнать, сколько (cnt) вакансий в каждом регионе (area).
Отсортируйте по количеству вакансий в порядке убывания.

In [13]:
query_4_1 = """
    SELECT 
        a.name AS region_name,
        COUNT(v.id) AS cnt
    FROM vacancies v
    JOIN areas a ON v.area_id = a.id
    GROUP BY a.id, a.name
    ORDER BY cnt DESC;
    """

In [14]:
df = pd.read_sql_query(query_4_1, connection)
n = 5
print(f"Топ-{n} регионов по количеству вакансий:")
for _, row in df.head(n).iterrows():
    print(f"{row['region_name']}: {row['cnt']}")

Топ-5 регионов по количеству вакансий:
Москва: 5333
Санкт-Петербург: 2851
Минск: 2112
Новосибирск: 2006
Алматы: 1892


### 4.2. Напишите запрос, чтобы определить у какого количества вакансий заполнено хотя бы одно из двух полей с зарплатой.

In [15]:
query_4_2 = """
    SELECT 
        COUNT(*) AS cnt_vacancies_with_salary
    FROM vacancies
    WHERE salary_from IS NOT NULL OR salary_to IS NOT NULL;
    """

In [16]:
df = pd.read_sql_query(query_4_2, connection)
print(f"Количество вакансий с заполненной зарплатой (хотя бы одно поле): {df.iloc[0, 0]}")

Количество вакансий с заполненной зарплатой (хотя бы одно поле): 24073


### 4.3. Найдите средние значения для нижней и верхней границы зарплатной вилки. Округлите значения до целого.

In [17]:
query_4_3 = """
    SELECT 
        ROUND(AVG(salary_from)) AS avg_salary_from,
        ROUND(AVG(salary_to)) AS avg_salary_to
    FROM vacancies
    WHERE salary_from IS NOT NULL OR salary_to IS NOT NULL;
    """

In [18]:
df = pd.read_sql_query(query_4_3, connection)
print(f"Средние значения зарплатной вилки (округленные до целого): от {df['avg_salary_from'].iloc[0]:.0f} до {df['avg_salary_to'].iloc[0]:.0f}")

Средние значения зарплатной вилки (округленные до целого): от 71065 до 110537


### 4.4. Напишите запрос, который выведет количество вакансий для каждого сочетания типа рабочего графика (schedule) и типа трудоустройства (employment), используемого в вакансиях. Результат отсортируйте по убыванию количества.

In [19]:
query_4_4 = """
    SELECT 
        schedule,
        employment,
        COUNT(*) AS cnt
    FROM vacancies
    GROUP BY schedule, employment
    ORDER BY cnt DESC;
    """

In [20]:
df = pd.read_sql_query(query_4_4, connection)
print("Количество вакансий по сочетанию графика работы и типа трудоустройства:")
for _, row in df.iterrows():
    print(f"{row['schedule']:<20} {row['employment']:<22} {row['cnt']:>6}")

Количество вакансий по сочетанию графика работы и типа трудоустройства:
Полный день          Полная занятость        35367
Удаленная работа     Полная занятость         7802
Гибкий график        Полная занятость         1593
Удаленная работа     Частичная занятость      1312
Сменный график       Полная занятость          940
Полный день          Стажировка                569
Вахтовый метод       Полная занятость          367
Полный день          Частичная занятость       347
Гибкий график        Частичная занятость       312
Полный день          Проектная работа          141
Удаленная работа     Проектная работа          133
Гибкий график        Стажировка                116
Сменный график       Частичная занятость       101
Удаленная работа     Стажировка                 64
Гибкий график        Проектная работа           18
Сменный график       Стажировка                 12
Вахтовый метод       Проектная работа            2
Сменный график       Проектная работа            1


### 4.5. Напишите запрос, выводящий значения поля "Требуемый опыт работы" (experience) в порядке возрастания количества вакансий, в которых указан данный вариант опыта. 

In [21]:
query_4_5 = """
    SELECT 
        experience,
        COUNT(*) AS cnt
    FROM vacancies
    GROUP BY experience
    ORDER BY cnt ASC;
    """

In [22]:
df = pd.read_sql_query(query_4_5, connection)
print("Количество вакансий по требуемому опыту (от меньшего к большему):")
for _, row in df.iterrows():
    print(f"{row['experience']:<25} {row['cnt']:>6}")

Количество вакансий по требуемому опыту (от меньшего к большему):
Более 6 лет                 1337
Нет опыта                   7197
От 3 до 6 лет              14511
От 1 года до 3 лет         26152


### Вывод по Юниту 4
1. Рынок сильно централизован — почти 29% всех вакансий сосредоточены в 5 крупнейших городах, при этом Москва занимает лидирующее положение.

2. Около половины вакансий содержат информацию о зарплате, что ограничивает возможности для полноценного анализа. Средняя зарплатная вилка составляет примерно 71–110 тыс. руб.

3. Классический формат (полный день + полная занятость) остаётся основным (более 70% вакансий). Однако заметен рост доли дистанционного формата — почти 16% вакансий.

4. Рынок ориентирован в основном на специалистов с опытом работы от 1 до 3-х лет. Возможности для начинающих специалистов ограничены, а для высококвалифицированных экспертов с опытом более 6 лет предложений крайне мало, что может свидетельствовать о дефиците таких специалистов, отсутствии у большинства работадателей достаточных финансовых возможностей для найма подобных кадров или об их поиске через другие каналы. 

В итоге, мы имеем дело с классическим рынком труда: фокус на крупные города, полный рабочий день и предложения для специалистов с опытом 1–3 года, со средним уровнем зарплатной прозрачности и заметным, но всё ещё вторичным сегментом удалённой занятости.

***

# Юнит 5. Анализ работодателей

### 5.1. Напишите запрос, который позволит узнать, какие работодатели находятся на первом и пятом месте по количеству вакансий.

In [23]:
query_5_1 = """
    WITH employer_vacancy_count AS (
        SELECT 
            e.id,
            e.name AS employer_name,
            COUNT(v.id) AS vacancy_count,
            ROW_NUMBER() OVER (ORDER BY COUNT(v.id) DESC, e.id) AS rank
        FROM employers e
        LEFT JOIN vacancies v ON e.id = v.employer_id
        GROUP BY e.id, e.name
    )
    SELECT 
        employer_name,
        vacancy_count,
        rank
    FROM employer_vacancy_count
    WHERE rank IN (1, 5)
    ORDER BY rank;
    """

In [24]:
df = pd.read_sql_query(query_5_1, connection)
print("Работодатели на 1-м и 5-м месте по количеству вакансий:")
for _, row in df.iterrows():
    print(f"{row['employer_name']:<20} {row['vacancy_count']:<12} {row['rank']:>4}")

Работодатели на 1-м и 5-м месте по количеству вакансий:
Яндекс               1933            1
Газпром нефть        331             5


### 5.2. Напишите запрос, который для каждого региона выведет количество работодателей и вакансий в нём.
Среди регионов, в которых нет вакансий, найдите тот, в котором наибольшее количество работодателей.

In [25]:
query_5_2=f'''
    SELECT a.name AS "название региона",
           count(e.id) AS "количество_работодателей",
           count(v.id) AS "количество_вакансий"
    FROM areas a
    LEFT JOIN vacancies v on a.id = v.area_id
    LEFT JOIN employers e on a.id = e.area
    GROUP BY a.name 
    HAVING count(v.id) = 0 
    ORDER BY 2 DESC
    '''

In [26]:
df = pd.read_sql_query(query_5_2, connection)
print(df[['название региона', 'количество_работодателей', 'количество_вакансий']].head(10).to_string(index=False))

     название региона  количество_работодателей  количество_вакансий
               Россия                       410                    0
            Казахстан                       207                    0
   Московская область                        75                    0
   Краснодарский край                        19                    0
   Ростовская область                        18                    0
             Беларусь                        18                    0
          Азербайджан                        17                    0
 Республика Татарстан                        16                    0
Нижегородская область                        16                    0
           Узбекистан                        15                    0


### 5.3. Для каждого работодателя посчитайте количество регионов, в которых он публикует свои вакансии. Отсортируйте результат по убыванию количества.

In [27]:
query_5_3 = """
    SELECT 
        e.id AS employer_id,
        e.name AS employer_name,
        COUNT(DISTINCT v.area_id) AS region_count
    FROM employers e
    JOIN vacancies v ON e.id = v.employer_id
    GROUP BY e.id, e.name
    ORDER BY region_count DESC, employer_name;
    """

In [28]:
df = pd.read_sql_query(query_5_3, connection)
print("Количество регионов, в которых каждый работодатель публикует вакансии (Топ-10):")
print("-" * 70)
for _, row in df.head(10).iterrows():
    print(f"{row['employer_name']:<35} {row['region_count']:>6}")
print("-" * 70)

Количество регионов, в которых каждый работодатель публикует вакансии (Топ-10):
----------------------------------------------------------------------
Яндекс                                 181
Ростелеком                             152
Спецремонт                             116
Поляков Денис Иванович                  88
ООО ЕФИН                                71
Совкомбанк                              63
МТС                                     55
ЭФКО, Управляющая компания              49
КРОН                                    48
Почта России                            48
----------------------------------------------------------------------


### 5.4. Напишите запрос для подсчёта количества работодателей, у которых не указана сфера деятельности. 

In [29]:
query_5_4 = """
    SELECT 
        COUNT(DISTINCT e.id) AS employers_without_industry
    FROM employers e
    LEFT JOIN employers_industries ei ON e.id = ei.employer_id
    WHERE ei.employer_id IS NULL;
    """

In [30]:
df = pd.read_sql_query(query_5_4, connection)
print(f"Количество работодателей без указанной сферы деятельности: {df.iloc[0, 0]}")

Количество работодателей без указанной сферы деятельности: 8419


### 5.5. Напишите запрос, чтобы узнать название компании, находящейся на третьем месте в алфавитном списке (по названию) компаний, у которых указано четыре сферы деятельности.

In [31]:
query_5_5 = """
    WITH companies_with_4_industries AS (
        SELECT 
            e.id,
            e.name AS employer_name,
            COUNT(DISTINCT ei.industry_id) AS industry_count
        FROM employers e
        JOIN employers_industries ei ON e.id = ei.employer_id
        GROUP BY e.id, e.name
        HAVING COUNT(DISTINCT ei.industry_id) = 4
    )
    SELECT 
        employer_name,
        industry_count,
        ROW_NUMBER() OVER (ORDER BY employer_name) AS alphabetical_rank
    FROM companies_with_4_industries
    ORDER BY employer_name
    LIMIT 1 OFFSET 2;
    """

In [32]:
df = pd.read_sql_query(query_5_5, connection)
company_name = df['employer_name'].iloc[0]
print(f"\nНазвание компании: {company_name}")
print(f"Количество сфер деятельности: {df['industry_count'].iloc[0]}")
print(f"Место в алфавитном списке: {df['alphabetical_rank'].iloc[0]}")


Название компании: 2ГИС
Количество сфер деятельности: 4
Место в алфавитном списке: 3


### 5.6. С помощью запроса выясните, у какого количества работодателей в качестве сферы деятельности указана Разработка программного обеспечения

In [33]:
query_5_6 = """
    SELECT 
        COUNT(DISTINCT e.id) AS employers_count
    FROM employers e
    JOIN employers_industries ei ON e.id = ei.employer_id
    JOIN industries i ON ei.industry_id = i.id
    WHERE i.name = 'Разработка программного обеспечения';
    """

In [34]:
df = pd.read_sql_query(query_5_6, connection)
employers_count = df['employers_count'].iloc[0]
print(f"\nКоличество работодателей у которых в сфере деятельности указана Разработка ПО: {employers_count}")


Количество работодателей у которых в сфере деятельности указана Разработка ПО: 3553


### 5.7. Для компании «Яндекс» выведите список регионов-миллионников, в которых представлены вакансии компании, вместе с количеством вакансий в этих регионах. 
Также добавьте строку Total с общим количеством вакансий компании. 
Результат отсортируйте по возрастанию количества.

Список городов-милионников надо взять [отсюда](https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8). 

Если возникнут трудности с этим задание посмотрите материалы модуля  PYTHON-17. Как получать данные из веб-источников и API. 

In [35]:
# GET-запрос на страницу с городами-миллионниками
response = requests.get('https://ru.wikipedia.org/wiki/Города-миллионеры_России', headers={'User-Agent': 'Mozilla/5.0'})

# проверка доступности страницы
print(response.status_code)

# получение таблиц из HTML-ответа
tables = pd.read_html(response.text, flavor='lxml')
# берём первую таблицу из всех, имеющихся на странице, забираем столбец "Город"
df = tables[0]
million_cities = df['Город'].tolist()

# очистка списка городов от пробелов, табуляций и символов новой строки, удаление сносок Википедии
million_cities = [city.split('[')[0].strip() for city in million_cities]

# занесение списка городов в кортеж
cities_tuple = tuple(million_cities)

# формируем SQL-запрос с учётом полученных городов-миллионников
query_5_7 = f"""
WITH yandex_employer AS (
    SELECT id 
    FROM employers 
    WHERE name ILIKE '%Яндекс%'
    LIMIT 1
),
yandex_vacancies AS (
    SELECT 
        v.id,
        v.area_id,
        a.name AS region_name
    FROM vacancies v
    JOIN areas a ON v.area_id = a.id
    WHERE v.employer_id = (SELECT id FROM yandex_employer)
)
SELECT 
    region_name,
    COUNT(*) AS vacancy_count
FROM yandex_vacancies
WHERE region_name IN {cities_tuple}
GROUP BY region_name
ORDER BY vacancy_count ASC;
"""

200


C:\Temp\ipykernel_10772\3218931323.py:4: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text, flavor='lxml')


In [36]:
# выполняем SQL-запрос и выводим результат

df_yandex = pd.read_sql_query(query_5_7, connection)

total_count = df_yandex['vacancy_count'].sum()
total_row = pd.DataFrame({'region_name': ['Total'], 'vacancy_count': [total_count]})
df_yandex_with_total = pd.concat([df_yandex, total_row], ignore_index=True)

print("Вакансии компании «Яндекс» в городах-миллионерах:")
print(df_yandex_with_total.to_string(index=False, header=False))

print(f"Количество регионов-миллионеров с вакансиями Яндекса: {len(df_yandex)}")

Вакансии компании «Яндекс» в городах-миллионерах:
           Омск  21
      Челябинск  22
     Красноярск  23
      Волгоград  24
          Пермь  25
 Ростов-на-Дону  25
         Казань  25
            Уфа  26
         Самара  26
      Краснодар  30
        Воронеж  32
    Новосибирск  35
Нижний Новгород  36
   Екатеринбург  39
Санкт-Петербург  42
         Москва  54
          Total 485
Количество регионов-миллионеров с вакансиями Яндекса: 16


### Вывод по Юниту 5
1. Яндекс является абсолютным лидером как по числу вакансий (1 933), так и по географическому охвату (181 регион), что подтверждает его статус крупнейшего работодателя на анализируемой платформе.

2. Существует значительное число регионов, где зарегистрированы работодатели, но нет активных вакансий. Это может свидетельствовать о неактивном использовании платформы, либо устаревших данных.

3. Помимо Яндекса, значительное региональное присутствие демонстрируют Ростелеком, Спецремонт, МТС и Почта России — компании с государственным участием или федерального масштаба.

4. Более трети работодателей не указали сферу деятельности, что ограничивает степень отраслевого анализа. При этом однозначно лидирует сфера «Разработка ПО».

5. Хотя Яндекс присутствует во всех городах-миллионниках, основная масса его вакансий (около 75%) находится за пределами крупнейших городов, что указывает на стратегию распределённого найма.

В базе представлены как крупные корпорации с охватом в тысячи вакансий и сотни регионов, так и небольшие компании и даже физические лица, что говорит о неоднородной структуре рынка. Это преимущественно крупные федерального значения компании с широкой географией, однако значительная доля работодателей имеет неполные или устаревшие данные. IT-сектор доминирует как по степени активности, так и по объему присутствия на рынке. Для более глубокого анализа может стать проблемой высокий процент отсутствующей информации и неактуальность части записей.

***

# Юнит 6. Предметный анализ

### 6.1. Сколько вакансий имеет отношение к данным?

Считаем, что вакансия имеет отношение к данным, если в её названии содержатся слова 'data' или 'данн'.

*Подсказка: Обратите внимание, что названия вакансий могут быть написаны в любом регистре.* 

In [37]:
query_6_1 = """
    SELECT 
        COUNT(*) as data_related_vacancies
    FROM vacancies
    WHERE LOWER(name) LIKE '%data%' 
       OR LOWER(name) LIKE '%данн%';
    """

In [38]:
df = pd.read_sql_query(query_6_1, connection)
data_vacancies = df['data_related_vacancies'].iloc[0]

print(f"Количество вакансий, имеющих отношение к данным: {data_vacancies}")

Количество вакансий, имеющих отношение к данным: 1771


### 6.2. Сколько есть подходящих вакансий для начинающего дата-сайентиста? 
Будем считать вакансиями для дата-сайентистов такие, в названии которых есть хотя бы одно из следующих сочетаний:
* 'data scientist'
* 'data science'
* 'исследователь данных'
* 'ML' (здесь не нужно брать вакансии по HTML)
* 'machine learning'
* 'машинн%обучен%'

** В следующих заданиях мы продолжим работать с вакансиями по этому условию.*

Считаем вакансиями для специалистов уровня Junior следующие:
* в названии есть слово 'junior' *или*
* требуемый опыт — Нет опыта *или*
* тип трудоустройства — Стажировка.

In [39]:
query_6_2 = """
    SELECT 
        COUNT(id) as junior_data_science_vacancies
    FROM vacancies
    WHERE (
        LOWER(name) LIKE '%data scientist%'
        OR LOWER(name) LIKE '%data science%'
        OR LOWER(name) LIKE '%исследователь данных%'
        OR LOWER(name) LIKE '%ml%' AND (LOWER(name) NOT LIKE '%html%')
        OR LOWER(name) LIKE '%machine learning%'
        OR LOWER(name) LIKE '%машинн%обучен%'
    )
    AND (
        LOWER(name) LIKE '%junior%'
        OR LOWER(experience) LIKE '%нет опыта%'
        OR LOWER(employment) LIKE '%стажировка%'
    );
    """

In [40]:
df = pd.read_sql_query(query_6_2, connection)
junior_ds = df['junior_data_science_vacancies'].iloc[0]
print(f"Количество вакансий для начинающего дата-сайентиста: {junior_ds}")

Количество вакансий для начинающего дата-сайентиста: 51


### 6.3. Сколько есть вакансий для DS, в которых в качестве ключевого навыка указан SQL или postgres?
** Критерии для отнесения вакансии к DS указаны в предыдущем задании.*

In [41]:
query_6_3 = f"""
    SELECT 
        COUNT(id) AS for_sql_and_postgres
    FROM vacancies
    WHERE
        (LOWER(name) LIKE '%data scientist%'
            OR LOWER(name) LIKE '%data science%'
            OR LOWER(name) LIKE '%исследователь данных%'
            OR LOWER(name) LIKE '%machine learning%'
            OR LOWER(name) LIKE '%машинн%обучен%'
            OR name LIKE '%ML%' AND LOWER(name) NOT LIKE '%html%')
    AND
    (LOWER(key_skills) LIKE '%sql%'
    OR LOWER(key_skills) LIKE '%postgres%'
    );
    """

In [42]:
df = pd.read_sql_query(query_6_3, connection)
ds_with_sql = df['for_sql_and_postgres'].iloc[0]
print(f"Количество вакансий для DS с SQL/postgres в ключевых навыках: {ds_with_sql}")

Количество вакансий для DS с SQL/postgres в ключевых навыках: 201


### 6.4. Проверьте, насколько популярен Python в требованиях работодателей к DS.Для этого вычислите количество вакансий, в которых в качестве ключевого навыка указан Python.

** Это можно сделать помощью запроса, аналогичного предыдущему.*

In [43]:
query_6_4 = f"""
    SELECT 
        COUNT(id) AS for_python
    FROM vacancies
    WHERE
        (LOWER(name) LIKE '%data scientist%'
            OR LOWER(name) LIKE '%data science%'
            OR LOWER(name) LIKE '%исследователь данных%'
            OR LOWER(name) LIKE '%machine learning%'
            OR LOWER(name) LIKE '%машинн%обучен%'
            OR name LIKE '%ML%' AND LOWER(name) NOT LIKE '%html%')
    AND
    (LOWER(key_skills) LIKE '%python%'
    );
    """

In [44]:
df = pd.read_sql_query(query_6_4, connection)
ds_pyton = df['for_python'].iloc[0]
print(f"Количество вакансий, в которых в качестве ключевого навыка указан Python: {ds_pyton}")

Количество вакансий, в которых в качестве ключевого навыка указан Python: 351


### 6.5. Сколько ключевых навыков в среднем указывают в вакансиях для DS?
Ответ округлите до двух знаков после точки-разделителя.

In [45]:
query_6_5 = """
    SELECT 
        ROUND(AVG(array_length(string_to_array(key_skills, '\t'), 1)), 2) as avg_skills_count
    FROM vacancies
    WHERE
        (LOWER(name) LIKE '%data scientist%'
            OR LOWER(name) LIKE '%data science%'
            OR LOWER(name) LIKE '%исследователь данных%'
            OR LOWER(name) LIKE '%machine learning%'
            OR LOWER(name) LIKE '%машинн%обучен%'
            OR name LIKE '%ML%' AND LOWER(name) NOT LIKE '%html%')
        AND key_skills IS NOT NULL
        AND key_skills != '';
    """

In [46]:
df = pd.read_sql_query(query_6_5, connection)
    
avg_skills = df['avg_skills_count'].iloc[0]
print(f"Среднее количество ключевых навыков в вакансиях для DS: {avg_skills}")

Среднее количество ключевых навыков в вакансиях для DS: 6.41


### 6.6. Напишите запрос, позволяющий вычислить, какую зарплату для DS в **среднем** указывают для каждого типа требуемого опыта (уникальное значение из поля *experience*). 

При решении задачи примите во внимание следующее:
1. Рассматриваем только вакансии, у которых заполнено хотя бы одно из двух полей с зарплатой.
2. Если заполнены оба поля с зарплатой, то считаем зарплату по каждой вакансии как сумму двух полей, делённую на 2. Если заполнено только одно из полей, то его и считаем зарплатой по вакансии.
3. Если в расчётах участвует null, в результате он тоже даст null (посмотрите, что возвращает запрос select 1 + null). Чтобы избежать этой ситуацию, мы воспользуемся функцией [coalesce](https://postgrespro.ru/docs/postgresql/9.5/functions-conditional#functions-coalesce-nvl-ifnull), которая заменит null на значение, которое мы передадим. Например, посмотрите, что возвращает запрос `select 1 + coalesce(null, 0)`

Выясните, на какую зарплату в среднем может рассчитывать дата-сайентист с опытом работы от 3 до 6 лет. Результат округлите до целого числа. 

In [47]:
query_6_6 = """
    SELECT 
        experience,
        ROUND(AVG(COALESCE(salary_from, salary_to, 0) + COALESCE(salary_to, salary_from, 0)) / 2.0) AS avg_salary_rounded
    FROM vacancies
    WHERE
        (LOWER(name) LIKE '%data scientist%'
            OR LOWER(name) LIKE '%data science%'
            OR LOWER(name) LIKE '%исследователь данных%'
            OR LOWER(name) LIKE '%machine learning%'
            OR LOWER(name) LIKE '%машинн%обучен%'
            OR name LIKE '%ML%' AND LOWER(name) NOT LIKE '%html%')
    AND (salary_from IS NOT NULL OR salary_to IS NOT NULL)
    GROUP BY experience
    ORDER BY avg_salary_rounded DESC;
    """

In [48]:
df = pd.read_sql_query(query_6_6, connection)

print("Средняя зарплата для DS-вакансий по типам опыта:")
print(df.rename(columns={'experience': 'опыт работы','avg_salary_rounded': 'средняя зарплата'}).to_string(index=False))

Средняя зарплата для DS-вакансий по типам опыта:
       опыт работы  средняя зарплата
     От 3 до 6 лет          243115.0
От 1 года до 3 лет          139675.0
         Нет опыта           74643.0


### Вывод по Юниту 6.
1. Мы имеем дело с активно развивающимся сегментом, имеющим большое количество вакансий для входа в профессию, широкий набор требуемых компетенций и чёткую зависимость дохода от стажа. 
2. Наибольший упор в требованиях делается на Python как основной инструмент, однако реальное количество компетенций кандидата должно быть значительно больше.
3. Рынок предлагает привлекательные зарплаты практически на всех уровнях, с высокими темпами их роста по мере накопления опыта, что делает сферу Data Science одной из наиболее перспективных для построения карьеры.

***

# Общий вывод по результатам анализа вакансий на HH.ru
## 1. Структура рынка
Рынок вакансий имеет ярко выраженную региональную концентрацию: Москва, Санкт-Петербург и другие города-миллионники имеют значительную долю предложений. 
При этом основная масса вакансий представлена в классическом формате "полный день + полная занятость", однако заметен устойчивый тренд на удалённую работу, что подтверждает трансформацию рынка труда в постпандемийный период.

## 2. Полнота данных
База данных характеризуется неравномерной полнотой заполнения:
Около половины вакансий содержат информацию о зарплате;
Более трети работодателей не указали сферу деятельности;
Поле ключевых навыков заполнено не для всех вакансий (данный момент требует осторожности при интерпретации данных о требуемых компетенциях).

Это в свою очередь указывает на необходимость предварительной очистки и нормализации данных перед проведением более глубокого анализа, а также на возможную потерю части информации, потенциально полезной для исследований.

## 3. IT-сектор
Компании IT-сферы, в особенности Яндекс, занимают лидирующие позиции как по количеству вакансий, так и по географическому охвату. 
Среди работодателей, указавших сферу деятельности, почти каждый четвёртый связан с разработкой ПО. 
Это подтверждает высокую цифровизацию российской экономики и доминирующую роль IT-компаний на рынке труда.

## 4. Рынок Data Science
Сектор Data Science насчитывает около 1,8 тыс. вакансий для начинающих специалистов, что составляет примерно 3,6% от общего числа вакансий. Профессия требует широкого набора компетенций (в среднем 6 навыков), при этом ключевым инструментом является Python. Наблюдается чёткая и значительная прогрессия заработной платы в зависимости от опыта:
* от 75 тыс. для новичков
* до ~140 тыс. через 1–3 года
* до ~243 тыс. при опыте 3–6 лет
Это делает сферу Data Science одной из наиболее привлекательных для построения карьеры с точки зрения как доступности входа, так и темпов финансового роста.

## 5. Ограничения анализа
При интерпретации результатов необходимо учитывать следующие ограничения:
* Неполнота данных (зарплаты, отрасли, навыки указаны не для всех записей);
* Возможная неактуальность части информации (регионы с работодателями, но без вакансий);
* Размытость части DS-вакансий (смежные позиции — аналитики, BI-специалисты и т.п.);
* Разная степень заполненности информационных полей работодателями.

## 6. Итоги
Анализируемая база данных отражает современный российский рынок труда со следующими ключевыми чертами:
* Высокая концентрация в крупнейших городах и IT-секторе;
* Активная трансформация форматов занятости (рост удалённой работы);
* Значительный спрос на специалистов с опытом 1–3 года с привлекательными перспективами;
* Привлекательные условия для входа в Data Science с быстрым ростом дохода.
* Данная структура позволяет проводить довольно содержательный анализ, но требует учёта ограничений, связанных с полнотой и качеством заполненной информации. Полученные результаты могут служить основой для принятия решений в области кадровой политики, выбора карьерной траектории или оценки рыночных тенденций.

# Дополнительно:

##  Выведем ТОП-10 наиболее востребованных навыков для Data Science специалистов.

In [49]:
query_add_1 = """
    SELECT 
        key_skill,
        COUNT(*) AS frequency
    FROM (
        SELECT 
            UNNEST(string_to_array(key_skills, '\t')) AS key_skill
        FROM vacancies
        WHERE LOWER(name) LIKE '%data scientist%'
           OR LOWER(name) LIKE '%data science%'
           OR LOWER(name) LIKE '%исследователь данных%'
           OR LOWER(name) LIKE '%machine learning%'
           OR LOWER(name) LIKE '%машинн%обучен%'
    ) AS skills
    GROUP BY key_skill
    ORDER BY frequency DESC
    LIMIT 10;
    """

In [50]:
df = pd.read_sql_query(query_add_1, connection)
print("Топ-10 наиболее востребованных в DS-вакансиях навыков:")
print(df.rename(columns={'key_skill': 'Навык','frequency': 'Частота встречаемости'}).to_string(index=False))

Топ-10 наиболее востребованных в DS-вакансиях навыков:
                    Навык  Частота встречаемости
                   Python                    289
                      SQL                    167
         Machine Learning                    102
Математическая статистика                     57
             Data Science                     52
            Data Analysis                     52
                   Pandas                     46
            Анализ данных                     43
          Английский язык                     40
    Статистический анализ                     40


### Вывод: 
Анализ частоты встречаемости навыков в DS-вакансиях демонстрирует нам ярко выраженное ядро обязательных компетенций:
1. Python (289 упоминаний) однозначно лидирует, что лишний раз подтверждает его статус **основного языка программирования** в индустрии. 
2. SQL (167) занимает второе место, подчёркивая критическую важность работы с базами данных, в том числе для специалистов по машинному обучению.
3. Machine Learning (102) находится на третьем месте, однако его отставание от Python и SQL говорит о том, что работодатели предпочитают универсальных специалистов, более узконаправленным ML-инженерам.

Также, обращает на себя внимание высокая доля навыков, связанных непосредственно с анализом данных и статистикой (суммарно с 6 позиций - 290). Это указывает на то, что в реальной работе DS-специалисту много времени уделяется исследовательскому анализу, проверке гипотез и интерпретации результатов, а не только построению моделей.

Присутствие в списке английского языка (40), что говорит о необходимости чтения документации, проф. литературы и коммуникации в международных командах.

# Проведем анализ потенциальных работодателей

In [51]:
query_add_2 = """
SELECT 
    e.name as employer_name,
    COUNT(*) as vacancies_count
FROM vacancies v
JOIN employers e ON v.employer_id = e.id
WHERE (LOWER(v.name) LIKE '%data scientist%'
    OR LOWER(v.name) LIKE '%data science%'
    OR LOWER(v.name) LIKE '%исследователь данных%'
    OR LOWER(v.name) LIKE '%machine learning%'
    OR LOWER(v.name) LIKE '%машинн%обучен%')
GROUP BY e.id, e.name
HAVING COUNT(*) >= 3
ORDER BY vacancies_count DESC;
"""

df = pd.read_sql_query(query_add_2, connection)

print(f"{'№':<3} {'Работодатель':<55} {'Количество вакансий':>10}")
for i, row in df.iterrows():
    print(f"{i+1:<3} {row['employer_name']:<55} {row['vacancies_count']:>10}")

№   Работодатель                                            Количество вакансий
1   СБЕР                                                            28
2   Банк ВТБ (ПАО)                                                  18
3   Bell Integrator                                                 14
4   VK                                                              12
5   Positive Technologies                                            9
6   Andersen                                                         7
7   МегаФон                                                          7
8   Бэнкс Софт Системс                                               6
9   Контур                                                           6
10  Ozon                                                             5
11  Газпромбанк                                                      5
12  2ГИС                                                             5
13  inDriver                                                        

## Вывод:
На основании имеющихся данных, можно утверждать, что мы имеем дело с высококонцентрированным, остродефицитным рынком, где около 40% спроса формируют всего 4 компании (СБЕР, ВТБ, Bell Integrator, VK), а основной фокус смещен с исследовательских задач в сторону внедрения ML, MLOps и интеграции. Джун-специалистам здесь практически нет места — спрос в основном на экспертов, способных не только построить модель, но и довести её до работающего бизнес-продукта в условиях импортозамещения и жестких требований к надёжности (особенно в сферах финансов и телекоммуникаций).



# ИТОГОВЫЙ ВЫВОД:
Свои дополнительные исследования базы данных я решил построить вокруг портрета "Идеального специалиста" Data Science / ML-инженера, сообразно потребностей рынка.
Совокупность данных рисует немного противоречивый, но довольно чёткий портрет искомого специалиста.
Итак, Подавляющее большинство работодателей интересует:
1. Крепкий middle+ (опыт 3–6 лет).
Это напрямую следует из двух фактов:
* зарплатная вилка для данной группы — ~243 тыс. руб. (максимальный темп роста);
* 40% спроса формируют крупные компании (СБЕР, ВТБ, VK) с высокими требованиями к надёжности, где недостаток компетентности может дорого обойтись компании.

2. Необходимые навыки:
* Python - базовый навык. Основа основ.
* SQL —  рабочий инструмент №2, так как данные в российских компаниях чаще всего лежат в реляционных БД.
* Machine Learning — присутствует, но не доминирует.
* Анализ и статистика (суммарно ~290 упоминаний) — это ключевой нюанс: до 50% рабочего времени уходит на исследование и обработку данных, проверку гипотез и визуализацию, а не на тренировку моделей.
* Английский язык (40 упоминаний) — необходим для чтения документации, библиотек и актуальных статей (особенно в условиях ухода западных компаний).

3. Что выходит за рамки «классического» DS.
Анализ сфер деятельности потенциальных работодателей показывает, что специалисту скорее всего будет необходимо:
* MLOps / инженерию — умение развернуть модель в продакшене;
* Понимание импортозамещения — опыт работы с российскими ОС, СУБД (PostgreSQL) и облачными платформами (VK Cloud, Yandex Cloud);
* Продуктовое мышление — способность сформулировать метрику успеха модели, как бизнес-показатель (снижение рисков, рост конверсии).
* Актуальные задачи (с точки зрения работодателя). Внедрение, а не исследование. Построить модель — лишь 20% задачи. Остальное — интеграция с корпоративным ландшафтом, мониторинг динамики данных, переобучение на новых данных.
* Работа в условиях неполных / "грязных данных". Поскольку вакансии содержат пропуски (зарплаты, отрасли), работодатель ожидает, что кандидат умеет принимать решения в условиях неопределённости и очищать данные по ходу работы. Отдельным пунктом стоит упомянуть "молодость" профессии. Многие работодатели недостаточно осведомлены о требованиях к правильному сбору и хранению данных.
* Коммуникация с бизнесом. Переводить требования отдела продаж или рисков на язык признаков и гипотез — критически важный навык (не отражён в списке, но становится логичным, исходя из структуры рынка.

4. Также хотелось бы заострить внимание на ограничениях, способных исказить понимание нашего "портрета":
* Неактуальность части информации (регионы с работодателями, но без вакансий).
* Отсутствие информации о динамике рынка. Вывод о «высокой концентрации у 4 компаний» — точечный факт. Однако без данных за последние 6–12 месяцев, нельзя с уверенностью утверждать, что это устойчивая структура, а не единомоментная потребность из-за крупных проектов.


В итоге, имеющиеся данные дают качественный, но статичный портрет. Для для понимания актуальности и динамики процессов крайне желательно:
1. провести повторный сбор данных через 6-12 месяцев для проверки устойчивости трендов;
2. выделить сезонные пики (например, активность найма в IT часто растёт в сентябре–октябре и падает в январе–феврале);
3. отдельно проанализировать вакансии с указанной зарплатой, чтобы исключить смещение (компании, которые скрывают вилку, часто предлагают ниже рынка).